# RF Cavity 3D — Maxwell (H alanı, Nédélec N0) · EigenspaceOperator3D

**Saf model tahmini:** ağ, her kenarda m=24 baz (vektör alan → kenar DOF) üretir → gradyan çekirdeği projekte edilmiş Rayleigh–Ritz → 6 mod (H alanı + frekans).

**Akış:** repo → ayarlar → kurulum → 3D veri (gmsh + N0 FEM) → PKL → eğitim (Drive'a checkpoint, kesilirse devam) → test değerlendirmesi (şekil tipi kırılımı).

Önce `MODE = "smoke"` ile deneyin, sonra `MODE = "full"`. GPU seçin (*Runtime → Change runtime type*).

In [ ]:
# ── 1. Repo ─────────────────────────────────────────────────────────────
import os
BRANCH = "claude/3d-maxwell"
REPO = "/content/rf_cavity_neural_operator"
if not os.path.exists(REPO):
    !git clone -q -b {BRANCH} https://github.com/KorayGokceler/rf_cavity_neural_operator.git {REPO}
%cd {REPO}
!git fetch -q origin {BRANCH} && git checkout -q {BRANCH} && git pull -q origin {BRANCH}
!git log --oneline -1

In [ ]:
# ── 2. Ayarlar ──────────────────────────────────────────────────────────
MODE = "smoke"            # "smoke": 12 geometri, 2 epoch | "full": gerçek eğitim
USE_DRIVE = True          # Colab dışı sunucuda False (Drive yok)
RESUME = True

if MODE == "smoke":
    N_TOTAL, MESH_SIZE, EPOCHS, BATCH = 12, 0.12, 2, 2
    EMBED_DIM, N_LAYERS = 32, 2
else:
    N_TOTAL = 3000        # ~2400 train / 300 val / 300 test
    MESH_SIZE = 0.10      # göreli tet boyutu → ~8k kenar DOF/geometri (0.07 → ~19k, daha doğru ama ~4× pahalı)
    EPOCHS, BATCH = 150, 8
    EMBED_DIM, N_LAYERS = 128, 4

FAMILIES = "pillbox axisym_cell blob"   # silindir, eksenel simetrik hücre, rastgele katı (deliksiz/kulpsuz)
N_MODES = 6               # saklanan / tahmin edilen mod sayısı
N_BASIS = 24              # Ritz deneme uzayı (≈ 2–3× mod)
LR = 2e-4
SEED = 0
NUM_WORKERS = min(8, os.cpu_count())
STORE_OPERATORS = N_TOTAL * 4.7e-3 * (0.10 / MESH_SIZE) ** 3 < 8   # GB tahmini (~4.7 MB/geometri @ mesh 0.10, ∝ h⁻³); büyükse M/K/G/Kp eğitimde yeniden kurulur

In [ ]:
# ── 3. Kurulum ──────────────────────────────────────────────────────────
!apt-get -qq install -y libglu1-mesa libxrender1 libxcursor1 libxft2 libxinerama1 > /dev/null
!pip -q install -r requirements.txt
import torch
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "YOK")

In [ ]:
# ── 4. Yollar ───────────────────────────────────────────────────────────
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = "/content/drive/MyDrive/rf_cavity_3d"
else:
    WORK = os.path.expanduser("~/rf_cavity_3d")
TAG  = f"{N_TOTAL}_h{MESH_SIZE}_seed{SEED}"
H5   = f"{WORK}/maxwell3d_{TAG}.h5"
PKL  = f"{WORK}/maxwell3d_{TAG}{'' if STORE_OPERATORS else '_noops'}.pkl"
LOGS = f"{WORK}/training_logs"
EXP  = f"eig3d_{MODE}_{TAG}_d{EMBED_DIM}_L{N_LAYERS}"
OUT  = f"{WORK}/results_{EXP}"
os.makedirs(OUT, exist_ok=True)
print("H5 :", H5); print("PKL:", PKL); print("EXP:", f"{LOGS}/{EXP}")

In [ ]:
# ── 5. 3D veri üretimi (gmsh + N0 FEM, H alanı; varsa atlanır) ────────────
# 4 çekirdekte ~1.2 s/geometri (mesh 0.10); çok çekirdekli sunucuda orantılı hızlanır.
if not os.path.exists(H5):
    !python src/data_gen/dataset_generator_3d.py --n_total {N_TOTAL} --n_eigen_modes {N_MODES} --mesh_size {MESH_SIZE} --families {FAMILIES} --seed {SEED} --n_workers {os.cpu_count()} --h5_filename {H5}
else:
    print("Mevcut:", H5)

In [ ]:
# ── 6. H5 → PKL ─────────────────────────────────────────────────────────
if not os.path.exists(PKL):
    !python convert_3d.py --h5_filepath {H5} --output_path {PKL} {'' if STORE_OPERATORS else '--no_operators'}
else:
    print("Mevcut:", PKL)

In [ ]:
# ── 7. Eğitim (kesilirse bu hücreyi tekrar çalıştırın → last.ckpt'den devam) ─
ckpt = f"{LOGS}/{EXP}/last.ckpt"
resume = f"--resume {ckpt}" if (RESUME and os.path.exists(ckpt)) else ""
print("Devam:", ckpt if resume else "yok (sıfırdan)")
!python train.py --config configs/eigenspace_3d.yaml {resume} --override dataset.data_path={PKL} dataset.random_seed={SEED} dataset.cache_operators={str(STORE_OPERATORS).lower()} model.embed_dim={EMBED_DIM} model.eigenspace.n_layers={N_LAYERS} model.n_basis={N_BASIS} model.num_field_modes={N_MODES} training.learning_rate={LR} training.max_epochs={EPOCHS} training.batch_size={BATCH} training.num_workers={NUM_WORKERS} training.log_dir={LOGS} training.exp_name={EXP}

In [ ]:
# ── 8. Test değerlendirmesi (saf model; şekil tipi kırılımı + CSV) ────────
!python scripts/eval_3d.py --checkpoint {LOGS}/{EXP} --data_path {PKL} --split test --csv {OUT}/test_metrics.csv

In [ ]:
# ── 9. (İsteğe bağlı) Eğitim eğrileri ───────────────────────────────────
%load_ext tensorboard
%tensorboard --logdir {LOGS}/{EXP}